# RF Vpi Sweep — Scanning Fabry-Perot Cavity

This notebook runs a frequency / power sweep of an RF source while
recording Fabry-Perot cavity transmission traces on an oscilloscope.
The analysis extracts the modulation index $\beta$ from sideband / carrier
area ratios and fits $V_\pi$ at each RF frequency.

## Workflow

1. **Configure** instruments + sweep grid.
2. **(Optional) Power calibration** — scope-based measurement of the actual RF
   voltage at each (frequency, power) point. Runs as a separate phase with
   its own timebase.
3. **Main sweep** — cavity trace acquisition and analysis. Uses calibrated
   voltages when available.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cavityscope.core import SweepConfig, PowerCalibration
from cavityscope.sweep import run_power_calibration, run_sweep
from cavityscope.analysis import reanalyze_with_calibration

## 1. Instrument initialisation

In [ ]:
from hardwarelib.oscilloscopes.rigol import RigolDHO4000
from hardwarelib.signal_generators.windfreak import WindfreakSynthHD

SCOPE_RESOURCE  = "USB0::0x1AB1::0x0610::HDO4A243800178::INSTR"
SCOPE_CHANNEL   = 1          # cavity photodetector
SCOPE_TIMEOUT   = 15_000     # ms

RF_COM_PORT     = "COM6"
RF_CHANNEL      = 0
RF_TIMEOUT      = 0.25       # s

scope = RigolDHO4000(SCOPE_RESOURCE, timeout_ms=SCOPE_TIMEOUT)
rf    = WindfreakSynthHD(RF_COM_PORT, channel=RF_CHANNEL, timeout_s=RF_TIMEOUT)

## 2. Sweep configuration

In [ ]:
cfg = SweepConfig(
    rf_frequencies_hz = np.linspace(954e6, 957e6, 21).tolist(),
    rf_powers_dbm     = np.linspace(-20.0, 4, 15).tolist(),

    cavity_fsr_hz         = 10e9,
    carrier_window_hz     = 120e6,
    sideband_window_hz    = 80e6,
    sideband_mode         = "minus",

    compute_vpi           = True,
    net_power_offset_db   = 0.0,
    assumed_load_ohm      = 50.0,

    output_dir            = "vpi_sweep_output",
    save_trace_plots      = True,
    save_reference_plots  = True,
    save_frequency_plots  = True,
    save_raw_traces_csv   = True,

    # --- Scope-based power calibration (optional) ---
    # Connect the RF output (after amplifier, before modulator) to this
    # scope channel with a high-impedance probe.  Set to None to skip.
    cal_scope_channel     = 2,      # set to None to skip calibration
    cal_cycles_to_capture = 20,
    cal_settle_s          = 0.15,
)

## 3. (Optional) Power calibration

Runs the same frequency/power grid but with the scope timebase set for the
RF frequency. Measures the actual Vpk via sine fit on the calibration
channel. Skip this cell if `cal_scope_channel = None`.

In [ ]:
cal = None

if cfg.cal_scope_channel is not None:
    try:
        scope.open()
        rf.open()
        cal = run_power_calibration(scope, rf, cfg)
    finally:
        try:
            rf.set_output(False)
        except Exception:
            pass
        rf.close()
        scope.close()

    print(f"\nCalibration: {cal}")
else:
    print("Scope calibration skipped (cal_scope_channel = None).")
    # Uncomment to load a previous calibration CSV instead:
    # cal = PowerCalibration.from_csv("path/to/power_calibration.csv")

In [ ]:
if cal is not None:
    cal_csv = sorted((
        p for p in __import__('pathlib').Path(cfg.output_dir).rglob('power_calibration.csv')
    ), key=lambda p: p.stat().st_mtime)[-1]
    cal_df = pd.read_csv(cal_csv)
    freqs = sorted(cal_df['frequency_hz'].unique())

    fig, ax = plt.subplots(figsize=(9, 4))
    for freq in freqs:
        sub = cal_df[cal_df['frequency_hz'] == freq].sort_values('power_dbm')
        ax.plot(sub['power_dbm'], sub['vpk_v'], 'o-', ms=4, lw=1.2,
                label=f"{freq/1e9:.2f} GHz")
    ax.set(xlabel='RF power setting (dBm)', ylabel='Measured Vpk (V)',
           title='Scope power calibration')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, ncol=max(1, len(freqs)//5))
    plt.tight_layout()
    plt.show()
else:
    print("No calibration data to plot.")

## 4. Main sweep

The scope timebase should be set for the cavity trace before running this.
If a calibration was measured above, it is passed in automatically.

In [ ]:
try:
    scope.open()
    rf.open()

    data = run_sweep(
        scope=scope,
        rf_source=rf,
        cfg=cfg,
        scope_channel=SCOPE_CHANNEL,
        calibration=cal,
    )
finally:
    try:
        rf.set_output(False)
    except Exception:
        pass
    rf.close()
    scope.close()

## 5. Inspect results

In [ ]:
df_results = data["results"]
df_fits    = data["fits"]

print(f"{len(df_results)} measurement points, {len(df_fits)} frequency fits")
if 'voltage_source' in df_results.columns:
    print(f"Voltage source: {df_results['voltage_source'].iloc[0]}")
df_fits

In [ ]:
if not df_fits.empty:
    freq_ghz = df_fits["rf_frequency_hz"] / 1e9
    vpi = df_fits["fit_vpi_v"]
    r2 = df_fits["fit_r2"]
    n_pts = df_fits["n_fit_points"]
    valid = np.isfinite(vpi)

    fig, (ax_vpi, ax_r2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    ax_vpi.plot(freq_ghz[valid], vpi[valid], "o-", markersize=7, lw=1.5, label="$V_\\pi$")
    for f, v, n in zip(freq_ghz[valid], vpi[valid], n_pts[valid]):
        ax_vpi.annotate(f"N={int(n)}", (f, v), textcoords="offset points",
                        xytext=(0, 8), fontsize=7, ha="center", alpha=0.7)
    if valid.any():
        idx_min = int(np.nanargmin(vpi[valid]))
        f_min = float(freq_ghz[valid].iloc[idx_min] if hasattr(freq_ghz[valid], 'iloc') else freq_ghz[valid][idx_min])
        v_min = float(vpi[valid].iloc[idx_min] if hasattr(vpi[valid], 'iloc') else vpi[valid][idx_min])
        ax_vpi.axvline(f_min, ls=":", lw=0.8, color="tab:red", alpha=0.6)
        ax_vpi.annotate(
            f"min $V_\\pi$ = {v_min:.3f} V\n@ {f_min*1e3:.2f} MHz",
            (f_min, v_min),
            textcoords="offset points", xytext=(12, -18),
            fontsize=8, color="tab:red", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="tab:red", lw=1.2),
        )
    ax_vpi.set_ylabel("$V_\\pi$ (V)")
    ax_vpi.set_title("$V_\\pi$ vs RF frequency")
    ax_vpi.grid(True, alpha=0.25)
    ax_vpi.legend()

    bar_w = 0.8 * float(np.min(np.diff(freq_ghz))) if len(freq_ghz) > 1 else 0.1
    ax_r2.bar(freq_ghz[valid], r2[valid], width=bar_w, alpha=0.7)
    ax_r2.set_ylim(0, 1.05)
    ax_r2.axhline(0.99, ls=":", lw=0.8, color="gray", alpha=0.5, label="R\u00b2=0.99")
    ax_r2.set(xlabel="RF frequency (GHz)", ylabel="Fit R\u00b2")
    ax_r2.grid(True, alpha=0.25)
    ax_r2.legend()

    plt.tight_layout()
    plt.show()

## 5b. (Optional) Re-analyze with a different power calibration

Use this to apply a calibration **after** the sweep has already been recorded,
or to swap in a new calibration CSV. Only the voltage mapping and Vpi fits
are recomputed — the raw sideband/carrier measurements stay unchanged.

In [ ]:
# --- Option A: load a previously saved calibration CSV ---
# cal_post = PowerCalibration.from_csv("path/to/power_calibration.csv")

# --- Option B: use the calibration from this session ---
# cal_post = cal

# --- Option C: reload sweep results from disk ---
# df_results = pd.read_csv("vpi_sweep_output/measurement_.../sweep_results.csv")

# Recompute with calibration (set calibration=None for analytical dBm→V)
data_recal = reanalyze_with_calibration(
    results_df=df_results,
    cfg=cfg,
    calibration=cal,         # swap in cal_post or None as needed
    output_dir=None,         # set a path to save updated CSVs and plots
)
df_results = data_recal["results"]
df_fits    = data_recal["fits"]
df_fits

## 6. Browse saved plots

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage

run_dir = sorted(Path(cfg.output_dir).glob("measurement_*"))[-1]

def show_plots(folder: Path, heading: str):
    pngs = sorted(folder.glob("*.png"))
    if not pngs:
        return
    print(f"\n{'='*60}\n{heading}  ({len(pngs)} plots)\n{'='*60}")
    for p in pngs:
        display(IPImage(filename=str(p), width=900))

for name, label in [
    ("vpi_vs_frequency.png", "Vpi vs frequency summary"),
    ("power_calibration.png", "Scope power calibration"),
]:
    p = Path(cfg.output_dir) / name
    if not p.exists():
        p = run_dir / name
    if p.exists():
        print(f"\n{'='*60}\n{label}\n{'='*60}")
        display(IPImage(filename=str(p), width=900))

show_plots(run_dir / "reference_plots", "Reference traces (time domain)")
show_plots(run_dir / "trace_plots",     "RF-on traces (time domain)")
show_plots(run_dir / "frequency_plots", "Traces in frequency space")
show_plots(run_dir / "fit_plots",       "Per-frequency Vpi fits")